# 01. Obtencion de la data

Esta notebook documenta la etapa de adquisicion de datos del caso de estudio. En esta version curada se conservan las dos fuentes principales del proyecto:

- vacantes laborales consultadas desde un snapshot local en `JSON`
- dataset de survey de Stack Overflow usado como base del analisis principal

## Alcance

- establecer rutas reproducibles dentro del proyecto curado
- cargar y validar ambas fuentes de datos
- generar salidas pequenas para trazabilidad de la adquisicion
- dejar listas las entradas para limpieza, EDA, visualizacion y dashboarding


## Subpaso 1. Configuracion de entorno

**Proposito:** establecer rutas reproducibles para trabajar desde la carpeta curada del proyecto.

**Resultado esperado:** acceso consistente a los archivos `data/jobs.json` y `data/survey_data_updated.csv`, junto con una carpeta comun de exportacion en `data/acquisition_outputs`.


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = DATA_DIR / "acquisition_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

JOBS_JSON_PATH = DATA_DIR / "jobs.json"
SURVEY_CSV_PATH = DATA_DIR / "survey_data_updated.csv"

JOBS_JSON_PATH, SURVEY_CSV_PATH, OUTPUT_DIR


## Fuente A. Vacantes laborales

Esta fuente representa la parte de adquisicion orientada a demanda de mercado. Para la version de portafolio se usa un snapshot local del dataset que originalmente se consultaba mediante una API del lab.


## Subpaso 2. Carga del dataset de vacantes

**Proposito:** leer el snapshot local del dataset usado en la etapa de API.

**Resultado:** una tabla base con vacantes y atributos relevantes para consultas por ubicacion y habilidades.

**Nota:** usar un snapshot local en lugar de depender de un servicio Flask en ejecucion mejora la reproducibilidad del repositorio.


In [ ]:
jobs_df = pd.read_json(JOBS_JSON_PATH)
jobs_df.head()


## Subpaso 3. Validacion rapida del dataset de vacantes

**Proposito:** confirmar tamano, columnas disponibles y presencia de nulos antes de generar conteos.

**Resultado:** el snapshot contiene `27,005` registros y `9` campos principales.

**Hallazgo breve:** la estructura ya es suficiente para responder preguntas de demanda por ubicacion y skills sin transformaciones complejas en esta etapa.


In [ ]:
jobs_schema_summary = pd.DataFrame(
    {
        "dtype": jobs_df.dtypes.astype(str),
        "missing_values": jobs_df.isna().sum(),
        "non_null_count": jobs_df.count(),
    }
)

print("Shape:", jobs_df.shape)
print("Columns:", jobs_df.columns.tolist())
jobs_schema_summary


## Subpaso 4. Helpers para consultas tipo API

**Proposito:** encapsular la logica de conteo por ubicacion y por tecnologia para que el proceso sea reutilizable.

**Resultado:** dos funciones sencillas permiten consultar el numero de vacantes por criterio.

**Hallazgo metodologico:** para tecnologias como `C` y `Java` conviene usar patrones mas estrictos que un `contains` ingenuo, porque de lo contrario se sobrecuentan coincidencias parciales.


In [ ]:
TECH_PATTERNS = {
    "C": r"(?:^|[^A-Za-z0-9])C(?:[^A-Za-z0-9]|$)",
    "C#": r"C#",
    "C++": r"C\+\+",
    "Java": r"(?:^|[^A-Za-z0-9])Java(?:[^A-Za-z0-9]|$)",
    "JavaScript": r"JavaScript",
    "Python": r"Python",
    "Scala": r"Scala",
    "Oracle": r"Oracle",
    "SQL Server": r"SQL Server",
    "MySQL Server": r"MySQL Server",
    "PostgreSQL": r"PostgreSQL",
    "MongoDB": r"MongoDB",
}

def get_number_of_jobs_by_location(location: str) -> int:
    return int(
        jobs_df["Location"].str.contains(location, case=False, na=False, regex=False).sum()
    )

def get_number_of_jobs_by_technology(technology: str) -> int:
    pattern = TECH_PATTERNS[technology]
    return int(
        jobs_df["Key Skills"].fillna("").str.contains(pattern, case=False, regex=True).sum()
    )

get_number_of_jobs_by_location("Seattle"), get_number_of_jobs_by_technology("Python")


## Subpaso 5. Conteo de vacantes por ubicacion

**Proposito:** medir la concentracion de ofertas en las ciudades objetivo del ejercicio.

**Resultado:** se genera una tabla resumida con el numero de vacantes por ubicacion.

**Hallazgos:** `Washington DC` lidera claramente el grupo analizado, seguido por `Detroit`, `Seattle` y `New York`. `Austin` y `San Francisco` aparecen bastante por debajo dentro de este subconjunto.


In [ ]:
locations = [
    "Los Angeles",
    "New York",
    "San Francisco",
    "Washington DC",
    "Seattle",
    "Austin",
    "Detroit",
]

location_counts = pd.DataFrame(
    {
        "Location": locations,
        "Number of Job Postings": [get_number_of_jobs_by_location(location) for location in locations],
    }
).sort_values("Number of Job Postings", ascending=False)

location_counts


## Subpaso 6. Conteo de vacantes por tecnologia

**Proposito:** estimar la demanda observada para un conjunto acotado de tecnologias.

**Resultado:** se obtiene una tabla comparable entre lenguajes, bases de datos y tecnologias pedidas en el ejercicio.

**Hallazgos:** `JavaScript` es la tecnologia con mayor numero de vacantes dentro de esta lista, `Python` aparece en un nivel intermedio y `MySQL Server` no presenta coincidencias en este snapshot.


In [ ]:
technologies = list(TECH_PATTERNS.keys())

technology_counts = pd.DataFrame(
    {
        "Technology": technologies,
        "Number of Job Postings": [get_number_of_jobs_by_technology(technology) for technology in technologies],
    }
).sort_values("Number of Job Postings", ascending=False)

technology_counts


## Subpaso 7. Exportacion de resultados de vacantes

**Proposito:** dejar salidas persistentes para visualizaciones, validacion manual y trazabilidad del proceso.

**Resultado:** la notebook exporta archivos `CSV` y `Excel` con conteos por ubicacion y tecnologia en `data/acquisition_outputs`.

**Hallazgo operativo:** separar esta adquisicion en archivos pequenos facilita reutilizar los conteos en notebooks posteriores sin releer el JSON completo cada vez.


In [ ]:
location_csv_path = OUTPUT_DIR / "job_postings_by_location.csv"
location_xlsx_path = OUTPUT_DIR / "job_postings_by_location.xlsx"
technology_csv_path = OUTPUT_DIR / "job_postings_by_technology.csv"
technology_xlsx_path = OUTPUT_DIR / "job_postings_by_technology.xlsx"
jobs_schema_csv_path = OUTPUT_DIR / "jobs_dataset_schema_summary.csv"

location_counts.to_csv(location_csv_path, index=False)
location_counts.to_excel(location_xlsx_path, index=False)
technology_counts.to_csv(technology_csv_path, index=False)
technology_counts.to_excel(technology_xlsx_path, index=False)
jobs_schema_summary.reset_index(names="column_name").to_csv(jobs_schema_csv_path, index=False)

print("Saved:")
print(location_csv_path)
print(location_xlsx_path)
print(technology_csv_path)
print(technology_xlsx_path)
print(jobs_schema_csv_path)


## Fuente B. Dataset survey de Stack Overflow

Esta es la fuente central del proyecto. En la carpeta curada se trabaja con `survey_data_updated.csv`, que funciona como snapshot de entrada para limpieza, EDA, visualizaciones y dashboarding.


## Subpaso 8. Carga del dataset survey

**Proposito:** incorporar el dataset principal de la encuesta para el resto del flujo analitico.

**Resultado:** se carga una tabla amplia con respuestas de desarrolladores, variables demograficas, stack tecnologico, compensacion, uso de AI y experiencia laboral.

**Nota:** esta adquisicion se documenta aqui para dejar trazabilidad del origen del dataset principal que luego sera limpiado y transformado.


In [ ]:
survey_df = pd.read_csv(SURVEY_CSV_PATH)
survey_df[["ResponseId", "MainBranch", "Age", "Employment", "Country"]].head()


## Subpaso 9. Validacion rapida del dataset survey

**Proposito:** revisar tamano, estructura y primeras senales de complejidad del dataset antes de pasar a limpieza.

**Resultado:** el archivo contiene `18,845` filas y `114` columnas.

**Hallazgos:** es un dataset de alta dimensionalidad, con varias columnas multiseleccion y abundantes valores faltantes, lo que justifica una etapa de limpieza dedicada antes del EDA.


In [ ]:
survey_schema_summary = pd.DataFrame(
    {
        "dtype": survey_df.dtypes.astype(str),
        "missing_values": survey_df.isna().sum(),
        "non_null_count": survey_df.count(),
    }
)

print("Shape:", survey_df.shape)
print("First 20 columns:", survey_df.columns[:20].tolist())
survey_schema_summary.head(20)


## Subpaso 10. Exportacion de metadata del survey

**Proposito:** persistir una vista compacta del esquema para apoyar la siguiente notebook de limpieza.

**Resultado:** se exporta un resumen de columnas, tipos y faltantes en `CSV`.

**Hallazgo operativo:** contar con esta metadata fuera de la notebook ayuda a priorizar columnas problematicas sin tener que releer visualmente un dataset tan ancho en cada revision.


In [ ]:
survey_schema_csv_path = OUTPUT_DIR / "survey_dataset_schema_summary.csv"
survey_preview_csv_path = OUTPUT_DIR / "survey_dataset_preview.csv"

survey_schema_summary.reset_index(names="column_name").to_csv(survey_schema_csv_path, index=False)
survey_df.head(10).to_csv(survey_preview_csv_path, index=False)

print("Saved:")
print(survey_schema_csv_path)
print(survey_preview_csv_path)


## Cierre de la etapa

Esta etapa deja listas las dos fuentes de datos del proyecto: una fuente de vacantes para senales de demanda y una fuente survey para el analisis principal. La siguiente notebook se concentrara en limpieza de datos y preparacion del dataset survey para analisis y dashboarding.
